Create boundary files and update cities lat lons.

In [ ]:
import os
import geopandas as gpd
import pandas as pd
import csv
import osmnx as ox

In [ ]:
import shapely
import shapely.ops as ops
def fill_holes(cov):
    """Fill holes (= shapely interiors) from a coverage Polygon or MultiPolygon
    """
    holeseq_per_poly = get_holes(cov)
    holes = []
    for hole_per_poly in holeseq_per_poly:
        for hole in hole_per_poly:
            holes.append(hole)
    eps = 0.00000001
    if isinstance(cov, shapely.geometry.multipolygon.MultiPolygon):
        cov_filled = ops.unary_union([poly for poly in cov] + [Polygon(hole).buffer(eps) for hole in holes])
    elif isinstance(cov, shapely.geometry.polygon.Polygon) and not cov.is_empty:
        cov_filled = ops.unary_union([cov] + [Polygon(hole).buffer(eps) for hole in holes])
    return cov_filled

def extract_relevant_polygon(placeid, mp):
    """Return the most relevant polygon of a multipolygon mp, for being considered the city limit.
    Depends on location.
    """
    if isinstance(mp, shapely.geometry.polygon.Polygon):
        return mp
    if placeid == "tokyo": # If Tokyo, take poly with most northern bound, otherwise largest
        p = max(mp.geoms, key=lambda a: a.bounds[-1])
    elif placeid == "reykjavik": # Southern part
        p = min(mp.geoms, key=lambda a: a.bounds[0])
    else:
        p = max(mp.geoms, key=lambda a: a.area)
    return p

def get_holes(cov):
    """Get holes (= shapely interiors) from a coverage Polygon or MultiPolygon
    """
    holes = []
    if isinstance(cov, shapely.geometry.multipolygon.MultiPolygon):
        for pol in cov.geoms: # cov is generally a MultiPolygon, so we iterate through its Polygons
            holes.append(pol.interiors)
    elif isinstance(cov, shapely.geometry.polygon.Polygon) and not cov.is_empty:
        holes.append(cov.interiors)
    return holes

In [ ]:
citiesaddedpathfile = '/Users/mszell/Github/BikeNetKit/dataexport/cities/development/european_addition1.csv'
outpath = '/Users/mszell/Tresorit/bikenetkitshare/boundaries/'

In [ ]:
cities = pd.read_csv(citiesaddedpathfile, sep=";")
cities.head()

In [ ]:
for r in cities.itertuples(index=True):
    filepath = outpath+r.cityid+".geojson"
    if not os.path.exists(filepath):
        location = ox.geocoder.geocode_to_gdf(r.nominatim_query)
        location = fill_holes(extract_relevant_polygon(r.cityid, shapely.geometry.shape(location['geometry'][0])))
        boundary = gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[location])
        boundary.to_file(filepath, driver="GeoJSON", RFC7946="YES")
    boundary = gpd.read_file(filepath)
    centroid = boundary.centroid
    cities.at[r.Index, 'center_lat'] = centroid[0].x
    cities.at[r.Index, 'center_lon'] = centroid[0].y
cities.sort_values('cityid', inplace=True)
cities.to_csv(citiesaddedpathfile, sep=';', index=False, float_format='%7.5f')